In [4]:
import pandas as pd
import glob
import os

# Path to the parent folder containing subfolders of csv files
base_path = "software/debug/sessions_files"

# Get all csv files from subfolders
csv_files = glob.glob(os.path.join(base_path, "**", "*.csv"), recursive=True)

# Dictionary to hold DataFrames with custom names
dataframes = {}

for file in csv_files:
    parent = os.path.dirname(file)
    if parent == base_path:   # skip files directly in the base folder
        continue
    else: 
        folder = os.path.basename(os.path.dirname(file))   # parent folder name
        filename = os.path.splitext(os.path.basename(file))[0]  # csv name without extension
        df_name = f"{folder}_{filename}"
    
        dataframes[df_name] = pd.read_csv(file)

# Example: access one DataFrame
keys = list(dataframes.keys())
print(keys)   # shows all dataframe names
print(dataframes[keys[0]].head())


['active_1', 'active_10', 'active_11', 'active_12', 'active_13', 'active_14', 'active_15', 'active_16', 'active_17', 'active_18', 'active_19', 'active_2', 'active_20', 'active_21', 'active_22', 'active_23', 'active_25', 'active_26', 'active_27', 'active_28', 'active_29', 'active_3', 'active_30', 'active_31', 'active_32', 'active_33', 'active_34', 'active_4', 'active_5', 'active_6', 'active_7', 'active_8', 'active_9', 'passive_1', 'passive_10', 'passive_11', 'passive_12', 'passive_13', 'passive_14', 'passive_15', 'passive_16', 'passive_17', 'passive_18', 'passive_19', 'passive_2', 'passive_20', 'passive_3', 'passive_4', 'passive_5', 'passive_6', 'passive_7', 'passive_8', 'passive_9']
   THRESHOLD   SCORE   UPPER ANGLE   UPPER THRESHOLD   UPPER_LOAD_CELL  \
0       1000       1             0              1000             32454   
1       1000       2             0              1000             32454   
2       1000       3             0              1000             32456   
3       1000

In [19]:
passive_6 = dataframes['passive_6'].copy()
passive_6.head()


,THRESHOLD,SCORE,UPPER ANGLE,UPPER THRESHOLD,UPPER_LOAD_CELL,LOWER ANGLE,LOWER THRESHOLD,LOWER_LOAD_CELL,DEP,DEM,TIMESTAMP,SAMPLE_NO
0,0,1,0,0,31998,111,0,37480,249,70,8/27/2025 3:42:20 PM,5
1,0,2,0,0,31998,111,0,37472,249,68,8/27/2025 3:42:20 PM,5
2,0,3,0,0,31999,111,0,37473,249,67,8/27/2025 3:42:20 PM,5
3,0,4,0,0,31999,111,0,37416,249,68,8/27/2025 3:42:20 PM,5
4,0,5,0,0,31997,111,0,37453,249,69,8/27/2025 3:42:20 PM,5


In [24]:
# Drop the unwanted columns
passive_6_keys = list(passive_6.columns)
# print(passive_6_keys)
cols_to_drop = [passive_6_keys[0], passive_6_keys[3], passive_6_keys[6], 
                passive_6_keys[8], passive_6_keys[9], passive_6_keys[10], passive_6_keys[11]]

passive_6 = passive_6.drop(columns=cols_to_drop, errors="ignore")

# Check the result
print(passive_6.head())

   SCORE  UPPER ANGLE  UPPER_LOAD_CELL  LOWER ANGLE  LOWER_LOAD_CELL
0      1            0            31998          111            37480
1      2            0            31998          111            37472
2      3            0            31999          111            37473
3      4            0            31999          111            37416
4      5            0            31997          111            37453


In [25]:
import numpy as np

def compute_A(l2, l3, theta1, theta2):
    """
    Compute vector A based on lengths l2, l3 and angles theta1, theta2.
    Angles must be in radians.
    """
    A = np.array([
        l2 * np.cos(theta2) + l3 * np.cos(theta1),
        l2 * np.sin(theta2) + l3 * np.sin(theta1)
    ])
    return A

In [33]:
l2 = 26
l3 = 46
lower_angles = passive_6[' LOWER ANGLE']
upper_angles = passive_6[' UPPER ANGLE']
theta1 = lower_angles[0]
theta2 = upper_angles[0]
# print(theta1 - theta2)

 # convert degrees to radians
theta1 = np.radians(theta1)
theta2 = np.radians(theta2)

A = compute_A(l2, l3, theta1, theta2)
print("A =", A)

A = [ 9.51507432 42.94469962]


In [36]:
x = A[0]
y = A[1]
x,y

(np.float64(9.515074320916188), np.float64(42.94469961887128))

In [37]:
import numpy as np

def compute_A(df, l2, l3):
    """
    Compute x and y for all rows in df and add them as new columns.
    df must contain columns ' LOWER ANGLE' and ' UPPER ANGLE' in degrees.
    """
    # Convert to radians
    theta1 = np.radians(df[' LOWER ANGLE'])
    theta2 = np.radians(df[' UPPER ANGLE'])

    # Compute x and y
    x = l2 * np.cos(theta2) + l3 * np.cos(theta1)
    y = l2 * np.sin(theta2) + l3 * np.sin(theta1)

    # Store in dataframe
    df['x'] = x
    df['y'] = y

    return df

# Constants
l2 = 26
l3 = 46

# Apply to passive_6
passive_6 = compute_A(passive_6, l2, l3)

print(passive_6.head())


    SCORE   UPPER ANGLE   UPPER_LOAD_CELL   LOWER ANGLE   LOWER_LOAD_CELL  \
0       1             0             31998           111             37480   
1       2             0             31998           111             37472   
2       3             0             31999           111             37473   
3       4             0             31999           111             37416   
4       5             0             31997           111             37453   

          x        y  
0  9.515074  42.9447  
1  9.515074  42.9447  
2  9.515074  42.9447  
3  9.515074  42.9447  
4  9.515074  42.9447  


In [41]:
import numpy as np
import pandas as pd

def process_dataframe(df, l2=26, l3=46):
    """
    Process a dataframe:
    1. Drop unwanted columns.
    2. Compute x and y from LOWER ANGLE and UPPER ANGLE.
    3. Return cleaned dataframe.
    """

    # Identify columns dynamically (based on your passive_6 example)
    cols_to_drop = [df.columns[0], df.columns[3], df.columns[6],
                    df.columns[8], df.columns[9], df.columns[10], df.columns[11]]
    # print("Columns to Drop: ", cols_to_drop) # uncomment for debugging
    
    df = df.drop(columns=cols_to_drop, errors="ignore")

    # Convert angles to radians
    theta1 = np.radians(df[' LOWER ANGLE'])
    theta2 = np.radians(df[' UPPER ANGLE'])

    # Compute x and y
    df['x'] = l2 * np.cos(theta2) + l3 * np.cos(theta1)
    df['y'] = l2 * np.sin(theta2) + l3 * np.sin(theta1)

    return df


# Example usage with passive_6
passive_6_processed = process_dataframe(dataframes['passive_6'].copy(), l2=26, l3=46)

print(passive_6_processed.head())


    SCORE   UPPER ANGLE   UPPER_LOAD_CELL   LOWER ANGLE   LOWER_LOAD_CELL  \
0       1             0             31998           111             37480   
1       2             0             31998           111             37472   
2       3             0             31999           111             37473   
3       4             0             31999           111             37416   
4       5             0             31997           111             37453   

          x        y  
0  9.515074  42.9447  
1  9.515074  42.9447  
2  9.515074  42.9447  
3  9.515074  42.9447  
4  9.515074  42.9447  


In [42]:
# Apply process_dataframe to all DataFrames in the dataframes dictionary
processed_dataframes = {}

for name, df in dataframes.items():
    processed_dataframes[name] = process_dataframe(df, l2=26, l3=46)

# Example: print keys and first few rows of one processed DataFrame
print(list(processed_dataframes.keys()))

['active_1', 'active_10', 'active_11', 'active_12', 'active_13', 'active_14', 'active_15', 'active_16', 'active_17', 'active_18', 'active_19', 'active_2', 'active_20', 'active_21', 'active_22', 'active_23', 'active_25', 'active_26', 'active_27', 'active_28', 'active_29', 'active_3', 'active_30', 'active_31', 'active_32', 'active_33', 'active_34', 'active_4', 'active_5', 'active_6', 'active_7', 'active_8', 'active_9', 'passive_1', 'passive_10', 'passive_11', 'passive_12', 'passive_13', 'passive_14', 'passive_15', 'passive_16', 'passive_17', 'passive_18', 'passive_19', 'passive_2', 'passive_20', 'passive_3', 'passive_4', 'passive_5', 'passive_6', 'passive_7', 'passive_8', 'passive_9']


In [43]:
processed_dataframes['active_1'].head()

,SCORE,UPPER ANGLE,UPPER_LOAD_CELL,LOWER ANGLE,LOWER_LOAD_CELL,x,y
0,1,0,32454,110,32871,10.267073,43.225861
1,2,0,32454,110,32907,10.267073,43.225861
2,3,0,32456,110,32919,10.267073,43.225861
3,4,0,32453,110,32921,10.267073,43.225861
4,5,0,32455,110,32924,10.267073,43.225861


In [45]:
max(processed_dataframes['active_1'][' SCORE'])

209

In [47]:
# Create a list to store results
results = []

for name, df in processed_dataframes.items():
    max_score = df[' SCORE'].max()
    # Assign reward based on rules
    if max_score > 150:
        reward = 3
    elif 100 < max_score <= 150:
        reward = 2
    elif 90 < max_score <= 100:
        reward = 1
    elif max_score == 90:
        reward = 0
    elif 70 < max_score < 90:
        reward = -1
    elif 50 < max_score <= 70:
        reward = -2
    else:  # max_score < 50
        reward = -3
    results.append({'df_name': name, 'max_score': max_score, 'reward': reward})

# Create a DataFrame from the results
reward_df = pd.DataFrame(results)

In [48]:
reward_df.head()

,df_name,max_score,reward
0,active_1,209,3
1,active_10,131,2
2,active_11,0,-3
3,active_12,7,-3
4,active_13,131,2


In [52]:
def final_processing(df):
    """
    Process a dataframe:
    1. Drop more unwanted columns.
    2. Return cleaned dataframe.
    """

    # Identify columns dynamically (based on your passive_6 example)
    cols_to_drop = [df.columns[0], df.columns[1], df.columns[2],
                    df.columns[3], df.columns[4]]
    # print("Columns to Drop: ", cols_to_drop) # uncomment for debugging
    
    df = df.drop(columns=cols_to_drop, errors="ignore")

    return df

In [53]:
final_processing(processed_dataframes['active_1'].copy())

,x,y
0,10.267073,43.225861
1,10.267073,43.225861
2,10.267073,43.225861
3,10.267073,43.225861
4,10.267073,43.225861
...,...,...
234,5.819089,42.251913
235,5.819089,42.251913
236,6.543721,42.597545
237,7.274276,42.930478


In [54]:
# Apply on all df
final_dataframes = {}
for name, df in processed_dataframes.items():
    final_dataframes[name] = final_processing(df)

In [55]:
final_dataframes['active_1'].head()

,x,y
0,10.267073,43.225861
1,10.267073,43.225861
2,10.267073,43.225861
3,10.267073,43.225861
4,10.267073,43.225861


In [56]:
# Save all processed DataFrames as CSV files
output_dir = "end_effector_coords"
os.makedirs(output_dir, exist_ok=True)

for name, df in final_dataframes.items():
    csv_path = os.path.join(output_dir, f"{name}.csv")
    df.to_csv(csv_path, index=False)

In [57]:
# save the rewards df as csv 

reward_df.to_csv("reward_df.csv", index=False)